In [50]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

In [51]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5))
])

training_dataset = ImageFolder(root=r"C:\Users\srija\OneDrive\Desktop\machine learning\DEEP LEARNING\data\training_set\training_set", transform=transform)
testing_dataset = ImageFolder(root=r"C:\Users\srija\OneDrive\Desktop\machine learning\DEEP LEARNING\data\test_set\test_set", transform=transform)

train_loader = DataLoader(training_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(testing_dataset, batch_size=32, shuffle=False)

for images, labels in train_loader:
    print(f"Images shape: {images.shape}")
    print(f"Labels shape: {labels.shape}") # check if data has been loaded.
    print(f"Classes: {training_dataset.classes}")
    print(f"Total samples: {len(training_dataset)}")
    break

Images shape: torch.Size([32, 3, 128, 128])
Labels shape: torch.Size([32])
Classes: ['cats', 'dogs']
Total samples: 8005


In [52]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(

            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),# in_channels = rgb(3), out_channesl=features detectors, padding=extra_gap
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)

        return x

In [53]:
model = CNN()
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)

In [ ]:
epochs = 20

for epoch in range(epochs):
    epoch_training_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        output = model.forward(images)
        loss = criteria(output, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()
    
    print(f"epoch={epoch+1}/{epochs}, and loss = {epoch_training_loss/len(train_loader)}")


epoch=1/20, and loss = 0.6757557684206867
epoch=2/20, and loss = 0.5850780444791117
epoch=3/20, and loss = 0.5173315031832433
epoch=4/20, and loss = 0.45773698466707513
epoch=5/20, and loss = 0.3953846656824963
epoch=6/20, and loss = 0.33363515601690075
epoch=7/20, and loss = 0.24889559476855266
epoch=8/20, and loss = 0.16846756717242092
epoch=9/20, and loss = 0.09833666791469675
epoch=10/20, and loss = 0.06742187471229183
epoch=11/20, and loss = 0.036768088184000944
epoch=12/20, and loss = 0.02474581349717149
epoch=13/20, and loss = 0.02826037684796875
epoch=14/20, and loss = 0.03505573723074679
epoch=15/20, and loss = 0.05533344774280649
epoch=16/20, and loss = 0.012100176822101108
epoch=17/20, and loss = 0.037684914599688986
epoch=18/20, and loss = 0.031684958313376574


In [ ]:
import numpy as np
all_preds = []
all_labels = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        output = model.forward(images)
        _, predicted = torch.max(output, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())

from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

accuracy = accuracy_score(all_labels, all_preds)
precisionScore = precision_score(all_labels, all_preds)
recallScore = recall_score(all_labels, all_preds)
confusionMatrix = confusion_matrix(all_labels, all_preds)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision Score: {precisionScore:.4f}")
print(f"Recall Score: {recallScore:.4f}")
print(f"Confusion Matrix: {confusionMatrix}")